# Makine Ogrenmesi Ara Sinav Odevi

Bu notebook, odevin tum adimlarini duzenli sekilde tamamlamak icin hazirlanmis baslangic iskeletidir.

## 1. Kutuphaneler

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

from xgboost import XGBClassifier
import shap

sns.set_theme(style='whitegrid')

## 2. Veri Setinin Yuklenmesi

Not: Asagidaki secimi tek veri seti ile devam edecek sekilde doldur.

In [ ]:
# dataset = load_breast_cancer()
dataset = load_wine()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')

display(X.head())
display(y.head())

## 3. Veri Seti Kalite Kontrolleri

In [ ]:
# Missing value kontrolu
missing_values = X.isnull().sum()
display(missing_values)

# Dtype bilgileri
display(X.dtypes)

## 4. Kesifsel Veri Analizi

In [ ]:
summary_df = pd.DataFrame({
    'mean': X.mean(),
    'median': X.median(),
    'min': X.min(),
    'max': X.max(),
    'std': X.std(),
    'q1': X.quantile(0.25),
    'q3': X.quantile(0.75)
})
display(summary_df)

In [ ]:
plt.figure(figsize=(12, 8))
corr = X.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Korelasyon Matrisi')
plt.show()

## 5. Scaling ve Veri Bolme

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.125, random_state=42, stratify=y_train_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_val.shape, X_test.shape)

## 6. PCA ve LDA

In [ ]:
pca_full = PCA()
pca_full.fit(X_train_scaled)

explained = pca_full.explained_variance_ratio_
threshold = explained.mean()
best_n_components = int((explained > threshold).sum())
best_n_components = max(best_n_components, 2)

pca = PCA(n_components=best_n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

lda = LinearDiscriminantAnalysis(n_components=2)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_val_lda = lda.transform(X_val_scaled)
X_test_lda = lda.transform(X_test_scaled)

print('PCA component sayisi:', best_n_components)
print('LDA component sayisi:', X_train_lda.shape[1])

## 7. Model Kurulumu

Bu bolumde ham veri, PCA verisi ve LDA verisi icin 5 model egitilecek.

In [ ]:
def evaluate_model(model, X_train_data, y_train_data, X_val_data, y_val_data):
    model.fit(X_train_data, y_train_data)
    y_pred = model.predict(X_val_data)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_val_data)
    else:
        y_score = None
    
    result = {
        'accuracy': accuracy_score(y_val_data, y_pred),
        'precision': precision_score(y_val_data, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_val_data, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_val_data, y_pred, average='weighted', zero_division=0)
    }
    
    if y_score is not None:
        try:
            result['roc_auc'] = roc_auc_score(y_val_data, y_score, multi_class='ovr')
        except Exception:
            result['roc_auc'] = np.nan
    else:
        result['roc_auc'] = np.nan
    
    return result

## 8. Sonuclar ve SHAP

Bu bolumde validation tablosu, en iyi modelin test sonuclari ve SHAP analizleri tamamlanacak.